# Step 2d: explaining the graph models

**Models (8), in two sub-groups**

| Sub-group | Models | What the model reads |
|---|---|---|
| **Graph neural networks (4)** | GCN, GAT, GraphSAGE, R-GCN | the 12 profile stats of each account **and** the interaction graph |
| **Structure only (4)** | node2vec + LR, DeepWalk + LR, label spreading, belief propagation | the interaction graph (plus 20% of the training labels for the last two) |

These are compared *within* their sub-group, not across: a GNN can use profile stats and the structure-only models cannot.

**Two explanation methods per model**

| Model | Method 1 | Method 2 | Explanation is about |
|---|---|---|---|
| GCN, GAT, GraphSAGE, R-GCN | GNNExplainer | Integrated Gradients | which **edges** (and which of the node's own features) drove the prediction for one account |
| node2vec, DeepWalk | SHAP | LIME | the 32 **embedding dimensions** (dimensions have no meaning of their own, see section 8) |
| Label spreading | seed contributions (closed form) | leave-one-seed-out | which **seed labels** drove the score |
| Belief propagation | incoming messages | leave-one-edge-out | which **neighbours / edges** drove the score |

GNNExplainer and Integrated Gradients come from PyTorch Geometric's `explain` module (the latter wraps Captum). Label spreading and belief propagation are not standard ML models, so SHAP / LIME do not apply directly; we use the model's own arithmetic and exact perturbation instead.

**The answer key**

* Trolls sit in **rings** with dense internal edges. For a troll, an edge inside its ring is the "right" explanation (`truth/edges_truth.csv`, `edge_type == "ring"`). Correctness is the AUC of edge importance for separating ring edges from the other edges the model could have used (0.5 = chance).
* Bots follow authentic accounts; humans have only background edges, so there is no planted edge truth for them. We still explain them, to look at the other criteria.
* For the GNNs we also check the account's **own features** against the planted tabular signal, as in the profile-stats notebook.

**Known confounds (not fixed)**: node *degree* alone separates the classes (AUC about 0.91), and belief propagation's score mostly tracks degree. A model can be right for the wrong reason, which is why section 3 measures how much each model relies on the graph and on degree.

**Criteria**: correctness, faithfulness (delete the top edges / seeds, compare with random), agreement, stability (small perturbation, plus a seed-repeat control). See `docs/step2_plan.md` for the definitions.

> Run top to bottom, one cell at a time. `QUICK = True` gives a few-minute smoke test. A full run takes roughly 30 to 60 minutes (GNNExplainer and belief propagation dominate).

## 0. Setup

In [ ]:
import sys
import time
import warnings
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))  # so `import xai_eval` works from notebooks/

import xai_eval  # noqa: F401  (imports xgboost / lightgbm before torch: macOS OpenMP workaround)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from xai_eval import data as xd
from xai_eval import io as xio
from xai_eval import loaders as L
from xai_eval import metrics as xm

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 160)
torch.set_num_threads(4)

import copy

import shap
import lime.lime_tabular
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import r2_score, roc_auc_score
from torch_geometric.explain import CaptumExplainer, Explainer, GNNExplainer
from torch_geometric.utils import k_hop_subgraph

In [ ]:
SEED = 0
QUICK = False

N_PER_KIND = 4 if QUICK else 50          # explained accounts per kind (GNNs, embeddings)
N_PER_KIND_PROP = 3 if QUICK else 30     # label spreading / belief propagation are costlier: use the first few of each kind
N_STAB_PER_KIND = 2 if QUICK else 15     # subset for the stability checks
N_STAB_PER_KIND_PROP = 2 if QUICK else 10
GNNEXP_EPOCHS = 20 if QUICK else 100     # optimisation steps of GNNExplainer per explained node
IG_STEPS = 20 if QUICK else 50           # Integrated Gradients steps
KS_EDGES = [1, 3, 5]                     # edges deleted in the faithfulness test (GNN, BP)
KS_SEEDS = [1, 2, 3, 5]                  # seeds deleted (label spreading)
KS_DIMS = [1, 2, 4, 8]                   # embedding dimensions deleted (node2vec, DeepWalk)
N_RANDOM = 5                             # random deletions averaged per instance (3 for belief propagation: it is slow)
N_RANDOM_BP = 2 if QUICK else 3
POOL = 30                                # label spreading: seeds considered = the 30 with the largest kernel weight
SHAP_NSAMPLES = 100 if QUICK else 500    # KernelSHAP samples (embedding models)
LIME_NSAMPLES = 300 if QUICK else 1000
NOISE_FRAC = 0.05                        # input noise for the stability test, as a share of each feature's std
FAMILY = "graph"
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Data, the explained sample, and the answer key

In [ ]:
D = xd.load_data(REPO_ROOT)
N, E_UND = D.n, len(D.src)               # accounts, undirected edges (edge ids below are positions in edges.csv)
FEATS = D.FEATURES

# Same sampling rule as the other notebooks, except that accounts with fewer than 2 edges are skipped: deleting
# "the top edges vs random edges" needs at least two edges to choose from.
S_IDX = D.sample_accounts(per_kind=N_PER_KIND, split="test", seed=SEED, min_degree=2)
KIND_S = D.kind[S_IDX]
P_POS = np.concatenate([np.flatnonzero(KIND_S == k)[:N_PER_KIND_PROP] for k in ("human", "bot", "troll")])  # propagation subset
STAB_POS = np.concatenate([np.flatnonzero(KIND_S == k)[:N_STAB_PER_KIND] for k in ("human", "bot", "troll")])
STAB_POS_P = np.concatenate([np.flatnonzero(KIND_S == k)[:N_STAB_PER_KIND_PROP] for k in ("human", "bot", "troll")])

# ANSWER KEY (never given to a model)
IS_RING = (D.edges_truth["edge_type"].to_numpy() == "ring")     # per undirected edge, same row order as edges.csv
RING_ID = D.truth["ring_id"].to_numpy()                         # -1 for non-trolls
TRUE_C = xd.true_feature_contributions(D, S_IDX)                # planted per-feature contribution of each explained account
DEG = np.bincount(np.r_[D.src, D.dst], minlength=N)

print(f"{len(S_IDX)} explained accounts ({N_PER_KIND} per kind); propagation models use {len(P_POS)} of them")
print(f"{IS_RING.sum()} of {E_UND} edges are ring edges; mean degree {DEG.mean():.1f}")
print("median degree by kind:", {k: int(np.median(DEG[D.kind == k])) for k in ("human", "bot", "troll")})

## 2. Reload the step-1 models and check them

The GNN weights, the node2vec / DeepWalk embeddings and their classifiers, and the seed sets of the two propagation models are loaded from step 1. Label spreading is rebuilt in closed form, `F = (1 - alpha) (I - alpha S)^-1 Y0` (step 1 iterated 100 times; the difference is about 1e-10), because the closed form makes the explanation exact: the class-1 mass at a node is a sum over seeds.

In [ ]:
GNNS = list(L.GNN_MODELS)                    # gcn, gat, graphsage, rgcn
X_ALL = torch.as_tensor(D.X_tab_std, dtype=torch.float32)
MODELS, CHECKS = {}, []
for name in GNNS:
    MODELS[name] = L.load_model(name, D)
    CHECKS.append(L.verify_reload(name, D, L.sigmoid(L.gnn_logits(MODELS[name], X_ALL, D.EDGE_INDEX))))
EMB, CLF = {}, {}
for name in L.EMBED_MODELS:
    MODELS[name] = L.load_model(name, D)
    EMB[name] = L.embedding_matrix(MODELS[name], N)
    CLF[name] = L.load_extra(name, D)["classifier"]
    CHECKS.append(L.verify_reload(name, D, CLF[name].predict_proba(EMB[name])[:, 1]))
MODELS["label_spreading"] = L.load_model("label_spreading", D)
LS = L.LabelSpreading(D, MODELS["label_spreading"])
CHECKS.append(L.verify_reload("label_spreading", D, LS.all_scores(), atol=1e-6))
MODELS["belief_propagation"] = L.load_model("belief_propagation", D)
BP0 = L.bp_scores(D, MODELS["belief_propagation"])
CHECKS.append(L.verify_reload("belief_propagation", D, BP0[0], atol=1e-8))
assert all(c["status"] == "OK" for c in CHECKS), "A reloaded model does not match its step-1 predictions"

## 3. Graph-reliance probe: how much does each model use the graph, and degree?

Three model-level facts that help to read the explanations:

* the **AUC** of the model on the test accounts, next to the AUC of a bare **degree** score (the confound);
* the rank correlation between the model's score and the account's degree;
* the AUC when **half of all edges are removed at random**. A model that leans heavily on the graph loses accuracy; a GNN that mostly reads the profile stats does not.

(The embedding models cannot be re-scored on a thinned graph without retraining, so they are skipped in the last column.)

In [ ]:
te = np.flatnonzero(D.is_te)
rng = np.random.default_rng(SEED)
KEEP_HALF = rng.random(E_UND) < 0.5                          # keep a random half of the undirected edges
EI_HALF = torch.as_tensor(np.vstack([np.r_[D.src[KEEP_HALF], D.dst[KEEP_HALF]], np.r_[D.dst[KEEP_HALF], D.src[KEEP_HALF]]]), dtype=torch.long)
LS_HALF = L.LabelSpreading(D, MODELS["label_spreading"], D.src[KEEP_HALF], D.dst[KEEP_HALF])

score_full = {n: L.sigmoid(L.gnn_logits(MODELS[n], X_ALL, D.EDGE_INDEX)) for n in GNNS}
score_half = {n: L.sigmoid(L.gnn_logits(MODELS[n], X_ALL, EI_HALF)) for n in GNNS}
for n in L.EMBED_MODELS:
    score_full[n], score_half[n] = CLF[n].predict_proba(EMB[n])[:, 1], None
score_full["label_spreading"], score_half["label_spreading"] = LS.all_scores(), LS_HALF.all_scores()
score_full["belief_propagation"] = BP0[0]
score_half["belief_propagation"] = L.bp_scores(D, MODELS["belief_propagation"], KEEP_HALF)[0]

rows = [dict(model="(degree alone)", auc=roc_auc_score(D.y[te], DEG[te]), spearman_with_degree=1.0, auc_half_edges=np.nan)]
for n, s in score_full.items():
    rows.append(dict(model=n, auc=roc_auc_score(D.y[te], s[te]),
                     spearman_with_degree=float(spearmanr(s[te], DEG[te])[0]),
                     auc_half_edges=np.nan if score_half[n] is None else roc_auc_score(D.y[te], score_half[n][te])))
PROBE = pd.DataFrame(rows).set_index("model")
display(PROBE.round(3))

How to read it: a model with a high `spearman_with_degree` is largely a degree detector, and a correct explanation of it will point at "this account has many edges" rather than at ring structure. A model whose AUC barely moves when half the edges are removed is not relying much on the graph, and then edge explanations are explaining something the model hardly uses.

## 4. Graph neural networks: explanation and evaluation code

For each explained account `v` we take the **3-hop neighbourhood** of `v` (two GNN layers need 2 hops; the third makes the degree normalisation of GCN exact) and explain the model's prediction there. The code checks that the sub-graph gives exactly the same logit as the full graph.

* **GNNExplainer** learns a soft mask over edges and over node features that keeps the prediction while being sparse. It explains the *class the model predicted* and is stochastic (random start).
* **Integrated Gradients** (through PyG's `CaptumExplainer`) integrates the gradient of the predicted probability with respect to an edge mask and a feature mask, from "everything off" to "everything on". Deterministic.

Both give an importance for every **edge that can influence `v`** (edges pointing into `v` or into one of its neighbours, because that is what two layers can see) and for the node's own 12 features. Edge importance is summed over the two directions of an undirected edge.

In [ ]:
class ProbOutput(torch.nn.Module):
    """Captum wants probabilities for binary node classification; GNNExplainer works on the raw logit."""

    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x, edge_index):
        return torch.sigmoid(self.m(x, edge_index))


def make_ctx(v):
    """Everything needed to explain / perturb node v: its 3-hop sub-graph and the edge bookkeeping."""
    subset, sub_ei, mapping, emask = k_hop_subgraph(int(v), 3, D.EDGE_INDEX, relabel_nodes=True, num_nodes=N)
    c = int(mapping)
    und = emask.nonzero().view(-1).numpy() % E_UND                # undirected id of every directed edge of the sub-graph
    nodes1 = torch.unique(torch.cat([torch.tensor([c]), sub_ei[0][sub_ei[1] == c]]))   # v and its neighbours
    relevant = torch.isin(sub_ei[1], nodes1).numpy()              # edges feeding v or a neighbour = what 2 layers can use
    uniq, inv = np.unique(und, return_inverse=True)
    rel_ids = np.unique(und[relevant])
    return dict(v=int(v), x=X_ALL[subset], ei=sub_ei, c=c, und=und, uniq=uniq, inv=inv,
                rel_ids=rel_ids, rel_pos=np.searchsorted(uniq, rel_ids), subset=subset)


def logit(name, ctx, removed=(), x=None):
    """Logit of node v with some undirected edges deleted (edge ids are global)."""
    keep = ~np.isin(ctx["und"], list(removed)) if len(removed) else slice(None)
    with torch.no_grad():
        return float(MODELS[name](ctx["x"] if x is None else x, ctx["ei"][:, keep])[ctx["c"]])


def explain_node(name, method, ctx, x=None, seed=SEED):
    """Returns (edge importance over ctx["rel_ids"], importance of v's own 12 features)."""
    torch.manual_seed(seed)
    # Work on a copy: PyG's explainers switch the message-passing layers into "explain" mode and a
    # GNNExplainer run leaves state behind that breaks the next Integrated Gradients call.
    net = copy.deepcopy(MODELS[name])
    if method == "GNNExplainer":
        algo, model, rt = GNNExplainer(epochs=GNNEXP_EPOCHS), net, "raw"
    else:
        algo, model, rt = CaptumExplainer("IntegratedGradients", n_steps=IG_STEPS), ProbOutput(net), "probs"
    ex = Explainer(model=model, algorithm=algo, explanation_type="model", node_mask_type="attributes",
                   edge_mask_type="object", model_config=dict(mode="binary_classification", task_level="node", return_type=rt))
    e = ex(ctx["x"] if x is None else x, ctx["ei"], index=ctx["c"])
    per_edge = np.bincount(ctx["inv"], weights=e.edge_mask.abs().detach().numpy(), minlength=len(ctx["uniq"]))
    return per_edge[ctx["rel_pos"]], e.node_mask[ctx["c"]].abs().detach().numpy()


GNN_METHODS = ["GNNExplainer", "Integrated Gradients"]
CTX = {}   # node -> context, built once and shared by all GNNs


def get_ctx(v):
    if v not in CTX:
        CTX[v] = make_ctx(v)
    return CTX[v]

In [ ]:
SC_GNN = xm.Scorecard("gnn")
SC_STRUCT = xm.Scorecard("structure_only")
INST_ROWS, STORE = [], {}


def eval_gnn(name):
    ctxs = [get_ctx(int(v)) for v in S_IDX]
    # sanity: the sub-graph reproduces the full-graph logit exactly
    full = L.gnn_logits(MODELS[name], X_ALL, D.EDGE_INDEX)
    assert np.allclose([logit(name, c) for c in ctxs[:20]], full[S_IDX[:20]], atol=1e-4), "sub-graph logit differs from the full graph"
    sd = float(np.std(full[D.is_te]))
    attrs, feats, secs = {}, {}, {}
    for m in GNN_METHODS:
        t0 = time.time()
        out = [explain_node(name, m, c) for c in ctxs]
        secs[m] = (time.time() - t0) / len(ctxs)
        attrs[m], feats[m] = [o[0] for o in out], np.stack([o[1] for o in out])
        print(f"  {m}: {secs[m]:.2f} s per account")
    n_items = np.array([len(c["rel_ids"]) for c in ctxs])
    troll = np.flatnonzero(KIND_S == "troll")
    for m in GNN_METHODS:
        # ---- correctness: AUC of edge importance for ring edges (trolls) ; own-feature rank correlation (all)
        auc = np.full(len(ctxs), np.nan)
        for i in troll:
            truth = IS_RING[ctxs[i]["rel_ids"]]
            if 0 < truth.sum() < len(truth):
                auc[i] = roc_auc_score(truth, attrs[m][i])
        SC_GNN.add(name, m, "correctness_ring_edge_auc", auc[troll], "chance = 0.5; trolls only")
        feat_rank = np.array([xm.rank_corr(feats[m][i], TRUE_C[i]) for i in range(len(ctxs))])
        SC_GNN.add(name, m, "correctness_own_feature_rank", feat_rank, "Spearman of own-feature importance with the planted contributions")
        # ---- faithfulness: delete top-k edges vs random k edges
        predict = lambda rows: np.array([logit(name, ctxs[i], rem) for i, rem in rows])  # noqa: E731
        remove = lambda x, idx: (x[0], tuple(ctxs[x[0]]["rel_ids"][idx]))                 # noqa: E731
        dele = xm.deletion_gain(predict, [(i, ()) for i in range(len(ctxs))], pad(attrs[m]), remove, KS_EDGES, sd,
                                n_random=N_RANDOM, seed=SEED, n_items=n_items)
        dele.assign(model=name, method=m).to_csv(xio.out_dir(D, FAMILY) / f"deletion_{name}_{m.replace(' ', '_')}.csv", index=False)
        faith = xm.mean_gain_per_instance(dele)
        SC_GNN.add(name, m, "faithfulness", faith)
        SC_GNN.add(name, m, "seconds_per_explanation", secs[m])
        for i in range(len(ctxs)):
            INST_ROWS.append(dict(model=name, method=m, account_id=int(S_IDX[i]), kind=KIND_S[i], ring_edge_auc=auc[i], faithfulness=faith[i]))
    # ---- agreement over the edges both methods rank
    a, b = attrs[GNN_METHODS[0]], attrs[GNN_METHODS[1]]
    label = " vs ".join(GNN_METHODS)
    SC_GNN.add(name, label, "agreement_spearman", [xm.rank_corr(x, y) for x, y in zip(a, b)])
    SC_GNN.add(name, label, "agreement_top3", [xm.topk_overlap(x, y, 3) for x, y in zip(a, b)])
    SC_GNN.add(name, label, "agreement_own_features", [xm.rank_corr(feats[GNN_METHODS[0]][i], feats[GNN_METHODS[1]][i]) for i in range(len(ctxs))])
    # ---- stability: small noise on every node's features; repeatability: same input, other seed
    for m in GNN_METHODS:
        st, rp = [], []
        for i in STAB_POS:
            c = ctxs[i]
            xn = c["x"] + NOISE_FRAC * torch.randn(c["x"].shape, generator=torch.Generator().manual_seed(SEED + int(i)))  # features are standardised: std = 1
            st.append(xm.rank_corr(attrs[m][i], explain_node(name, m, c, x=xn)[0]))
            rp.append(xm.rank_corr(attrs[m][i], explain_node(name, m, c, seed=SEED + 100)[0]))
        SC_GNN.add(name, m, "stability_noise", st)
        SC_GNN.add(name, m, "repeatability_seed", rp, "1.0 expected for a deterministic method")
    STORE[name] = dict(attrs=attrs, ctxs=ctxs)
    recs = [dict(model=name, method=m, account_id=int(S_IDX[i]), kind=KIND_S[i], edge_id=int(e), is_ring=bool(IS_RING[e]), importance=float(w))
            for m in GNN_METHODS for i in range(len(ctxs)) for e, w in zip(ctxs[i]["rel_ids"], attrs[m][i])]
    xio.save_table(D, FAMILY, f"attributions_{name}", pd.DataFrame(recs))


def pad(vectors):
    d = max(len(v) for v in vectors)
    return np.stack([np.pad(np.asarray(v, float), (0, d - len(v))) for v in vectors])


def show_gnn(name):
    i = int(np.flatnonzero(KIND_S == "troll")[0])
    c = STORE[name]["ctxs"][i]
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.2), sharey=False)
    for ax, m in zip(axes, GNN_METHODS):
        w = STORE[name]["attrs"][m][i]
        order = np.argsort(-w)
        ax.bar(np.arange(len(w)), w[order], color=["#E45756" if IS_RING[e] else "#BBBBBB" for e in c["rel_ids"][order]], width=1.0)
        ax.set_title(f"{name} / {m}: edges around one troll, sorted (red = ring edge)", fontsize=9)
        ax.set_xlabel("edge rank")
        ax.set_ylabel("importance")
    plt.tight_layout()
    plt.show()
    keep = ["correctness_ring_edge_auc", "correctness_own_feature_rank", "faithfulness", "agreement_spearman", "stability_noise", "repeatability_seed"]
    display(SC_GNN.df[(SC_GNN.df.model == name) & SC_GNN.df.criterion.isin(keep)].pivot_table(index="method", columns="criterion", values="value").round(3))

## 5. GCN

Each account's representation is a degree-normalised average of its neighbours' features, twice. Ring members share many neighbours, so edges inside the ring should matter for a troll. The probe in section 3 tells us whether this model really uses the graph.

In [ ]:
print('=== gcn ==='); eval_gnn('gcn'); show_gnn('gcn')

## 6. GAT

Like the GCN, but each neighbour gets a learned attention weight, so edges can matter very unequally. Edge-level explanations should be sharper here, if the attention picked up the ring.

In [ ]:
print('=== gat ==='); eval_gnn('gat'); show_gnn('gat')

## 7. GraphSAGE and R-GCN

### 7.1 GraphSAGE

Keeps the node's own features separate from the **mean** of its neighbours' features, so a node's own profile and its neighbourhood enter through different weights.

In [ ]:
print('=== graphsage ==='); eval_gnn('graphsage'); show_gnn('graphsage')

### 7.2 R-GCN

Relational GCN. In this data there is only one relation (an undirected interaction), so R-GCN is essentially a GCN with a separate self-connection weight; the explanation questions are the same as for the GCN.

In [ ]:
print('=== rgcn ==='); eval_gnn('rgcn'); show_gnn('rgcn')

## 8. Structure-only model 1 and 2: node2vec and DeepWalk (+ logistic regression)

Each account is represented by a 32-dimensional vector learned from random walks on the graph (node2vec biases the walks; DeepWalk is node2vec with p = q = 1), and a logistic regression sits on top. We explain the classifier **in terms of the 32 embedding dimensions** with SHAP (KernelSHAP) and LIME.

This is where explanation hits a wall: embedding dimensions have no meaning of their own, so there is no "right" dimension to find and **correctness is not defined**. We report faithfulness, agreement and stability, and then run a *probe* that asks what the dimensions encode (degree? ring membership?). That is the closest an embedding model gets to an explanation a person can use.

In [ ]:
EMB_METHODS = ["SHAP", "LIME"]
BG_EMB = {n: shap.kmeans(EMB[n][L.make_background(D, k=100, seed=SEED)], 10) for n in L.EMBED_MODELS}
TR_MEAN = {n: EMB[n][D.is_tr].mean(0) for n in L.EMBED_MODELS}
TR_SD = {n: EMB[n][D.is_tr].std(0) for n in L.EMBED_MODELS}


def emb_explain(name, method, Z, seed=SEED):
    f = lambda A: CLF[name].predict_proba(np.asarray(A))[:, 1]  # noqa: E731
    t0 = time.time()
    if method == "SHAP":
        np.random.seed(seed)
        A = np.asarray(shap.KernelExplainer(f, BG_EMB[name]).shap_values(Z, nsamples=SHAP_NSAMPLES, silent=True))
    else:
        ex = lime.lime_tabular.LimeTabularExplainer(EMB[name][D.is_tr], mode="regression", random_state=seed,
                                                    feature_names=[f"d{j}" for j in range(Z.shape[1])])
        A = np.zeros_like(Z)
        for i, z in enumerate(Z):
            e = ex.explain_instance(z, f, num_features=Z.shape[1], num_samples=LIME_NSAMPLES)
            for j, w in next(iter(e.local_exp.values())):
                A[i, j] = w
    return A, (time.time() - t0) / len(Z)


def eval_embedding(name):
    Z = EMB[name][S_IDX]
    f = lambda A: CLF[name].predict_proba(np.asarray(A))[:, 1]  # noqa: E731
    sd = float(np.std(f(EMB[name][D.is_te])))
    attrs, secs = {}, {}
    for m in EMB_METHODS:
        attrs[m], secs[m] = emb_explain(name, m, Z)
        print(f"  {m}: {secs[m]:.2f} s per account")
    def remove(x, idx):
        x = x.copy()
        x[idx] = TR_MEAN[name][idx]
        return x
    for m in EMB_METHODS:
        dele = xm.deletion_gain(lambda rows: f(np.vstack(rows)), list(Z), attrs[m], remove, KS_DIMS, sd, n_random=N_RANDOM, seed=SEED)
        dele.assign(model=name, method=m).to_csv(xio.out_dir(D, FAMILY) / f"deletion_{name}_{m}.csv", index=False)
        SC_STRUCT.add(name, m, "faithfulness", xm.mean_gain_per_instance(dele))
        SC_STRUCT.add(name, m, "seconds_per_explanation", secs[m])
        Zs = Z[STAB_POS]
        noisy = xm.noisy_copy(Zs, TR_SD[name], frac=NOISE_FRAC, seed=SEED + 1)
        SC_STRUCT.add(name, m, "stability_noise", xm.stability(np.abs(attrs[m][STAB_POS]), np.abs(emb_explain(name, m, noisy)[0])))
        SC_STRUCT.add(name, m, "repeatability_seed", xm.stability(np.abs(attrs[m][STAB_POS]), np.abs(emb_explain(name, m, Zs, seed=SEED + 100)[0])))
    a, b = np.abs(attrs["SHAP"]), np.abs(attrs["LIME"])
    SC_STRUCT.add(name, "SHAP vs LIME", "agreement_spearman", [xm.rank_corr(x, y) for x, y in zip(a, b)])
    SC_STRUCT.add(name, "SHAP vs LIME", "agreement_top5", [xm.topk_overlap(x, y, 5) for x, y in zip(a, b)])
    STORE[name] = dict(attrs=attrs)
    display(SC_STRUCT.df[SC_STRUCT.df.model == name].pivot_table(index="method", columns="criterion", values="value").round(3))

### 8.1 node2vec + logistic regression

In [ ]:
print('=== node2vec_logreg ==='); eval_embedding('node2vec_logreg')

### 8.2 DeepWalk + logistic regression

In [ ]:
print('=== deepwalk_logreg ==='); eval_embedding('deepwalk_logreg')

### 8.3 What do the embedding dimensions encode? (probe)

On the training accounts we fit a ridge regression from the 32 dimensions to `log(1 + degree)` and a logistic regression to "is a troll (ring member)". We report the held-out test R² and AUC. If the dimensions encode degree almost perfectly, then any explanation in terms of dimensions is, in disguise, an explanation in terms of degree.

In [ ]:
rows = []
is_troll = (D.kind == "troll").astype(int)
for n in L.EMBED_MODELS:
    E = EMB[n]
    r2 = r2_score(np.log1p(DEG[D.is_te]), Ridge(alpha=1.0).fit(E[D.is_tr], np.log1p(DEG[D.is_tr])).predict(E[D.is_te]))
    auc = roc_auc_score(is_troll[D.is_te], LogisticRegression(max_iter=2000).fit(E[D.is_tr], is_troll[D.is_tr]).predict_proba(E[D.is_te])[:, 1])
    rows.append(dict(model=n, r2_predicting_log_degree=r2, auc_predicting_troll_ring_membership=auc))
EMB_PROBE = pd.DataFrame(rows).set_index("model")
display(EMB_PROBE.round(3))

## 9. Structure-only model 3: label spreading

20% of the training accounts have their label revealed (the **seeds**). The label then spreads along edges, and an account's score is the share of "inauthentic" label mass that reaches it. Because the spreading is linear, the score of account `v` is exactly a sum over seeds, and we can explain it exactly.

* **Method 1, contribution:** seed `s` contributes `K[v,s] * (label_s - score_v)`, where `K[v,s]` is how much of `s`'s label reaches `v`. Contributions add up to zero around the score: seeds labelled inauthentic push it up, seeds labelled authentic push it down.
* **Method 2, leave-one-seed-out:** remove seed `s`, recompute the score, report the change. Exact, because the closed form needs no re-run.

Only the 30 seeds with the largest kernel weight for `v` are considered (the pool); the rest carry a negligible share, which the next cell reports.

**Do not over-read the agreement row.** Removing seed `s` changes the score by `K[v,s] * (label_s - score_v)` divided by a normaliser that is nearly the same for every seed, so the two methods rank seeds almost identically by construction. Agreement near 1 is expected and says little. The informative columns are correctness and faithfulness.

**Correctness:** do the top-3 seeds belong to the same ring as the explained troll? (Only trolls that have at least one same-ring seed in the pool are scored; chance is the same-ring share of the pool.)

In [ ]:
P_NODES = [int(v) for v in S_IDX[P_POS]]
K = {v: LS.kernel_row(v) for v in P_NODES}            # one linear solve per explained account
SEEDS = LS.seeds
LS_SCORES = LS.all_scores()
LS_SD = float(np.std(LS_SCORES[D.is_te]))
POOLS = {v: SEEDS[np.argsort(-K[v][SEEDS])[:POOL]] for v in P_NODES}
cover = [K[v][POOLS[v]].sum() / K[v][SEEDS].sum() for v in P_NODES]
print(f"the {POOL}-seed pool covers {np.mean(cover):.0%} of the label mass that reaches an account (mean over explained accounts)")


def ls_attr(v, pool, removed=()):
    s_v = LS.score_from_row(K[v], removed)
    contrib = K[v][pool] * (LS.Y0[pool, 1] - s_v)
    loo = np.array([s_v - LS.score_from_row(K[v], tuple(removed) + (int(p),)) for p in pool])
    return contrib, loo


def eval_label_spreading():
    kinds = KIND_S[P_POS]
    A = {"Seed contribution": [], "Leave-one-seed-out": []}
    t0 = time.time()
    for v in P_NODES:
        c, l = ls_attr(v, POOLS[v])
        A["Seed contribution"].append(c)
        A["Leave-one-seed-out"].append(l)
    secs = (time.time() - t0) / len(P_NODES)
    n_items = np.array([len(POOLS[v]) for v in P_NODES])
    predict = lambda rows: np.array([LS.score_from_row(K[P_NODES[i]], rem) for i, rem in rows])  # noqa: E731
    remove = lambda x, idx: (x[0], tuple(int(s) for s in POOLS[P_NODES[x[0]]][idx]))             # noqa: E731
    for m, vecs in A.items():
        prec = []
        for i, v in enumerate(P_NODES):
            if kinds[i] == "troll":
                same = RING_ID[POOLS[v]] == RING_ID[v]
                prec.append(xm.top_k_precision(vecs[i], same, 3) if same.any() else np.nan)
        SC_STRUCT.add("label_spreading", m, "correctness_same_ring_top3", prec, "trolls with a same-ring seed in the pool; chance = same-ring share of the pool")
        dele = xm.deletion_gain(predict, [(i, ()) for i in range(len(P_NODES))], pad(vecs), remove, KS_SEEDS, LS_SD, n_random=N_RANDOM, seed=SEED, n_items=n_items)
        dele.assign(model="label_spreading", method=m).to_csv(xio.out_dir(D, FAMILY) / f"deletion_label_spreading_{m.replace(' ', '_')}.csv", index=False)
        SC_STRUCT.add("label_spreading", m, "faithfulness", xm.mean_gain_per_instance(dele))
        SC_STRUCT.add("label_spreading", m, "seconds_per_explanation", secs)
    a, b = A["Seed contribution"], A["Leave-one-seed-out"]
    SC_STRUCT.add("label_spreading", "contribution vs leave-one-out", "agreement_spearman", [xm.rank_corr(np.abs(x), np.abs(y)) for x, y in zip(a, b)])
    SC_STRUCT.add("label_spreading", "contribution vs leave-one-out", "agreement_top3", [xm.topk_overlap(x, y, 3) for x, y in zip(a, b)])
    # ---- stability: drop one random pool seed outside the top-2, re-explain, compare on the seeds that remain
    r = np.random.default_rng(SEED + 5)
    st = {m: [] for m in A}
    for i in np.flatnonzero(np.isin(P_POS, STAB_POS_P)):
        v, pool = P_NODES[i], POOLS[P_NODES[i]]
        top2 = set(np.argsort(-np.abs(A["Seed contribution"][i]))[:2])
        j = int(r.choice([q for q in range(len(pool)) if q not in top2]))
        keep = [q for q in range(len(pool)) if q != j]
        c2, l2 = ls_attr(v, pool[keep], removed=(int(pool[j]),))
        st["Seed contribution"].append(xm.rank_corr(np.abs(A["Seed contribution"][i][keep]), np.abs(c2)))
        st["Leave-one-seed-out"].append(xm.rank_corr(np.abs(A["Leave-one-seed-out"][i][keep]), np.abs(l2)))
    for m, v in st.items():
        SC_STRUCT.add("label_spreading", m, "stability_drop_item", v, "drop one seed outside the top 2; both methods are deterministic")
    STORE["label_spreading"] = dict(attrs=A)
    display(SC_STRUCT.df[SC_STRUCT.df.model == "label_spreading"].pivot_table(index="method", columns="criterion", values="value").round(3))

In [ ]:
print('=== label_spreading ==='); eval_label_spreading()

## 10. Structure-only model 4: belief propagation

Every edge carries a **message** in each direction ("my neighbour thinks you are probably a bot"); an account's belief combines its own prior (confident only for seeds) with the messages it receives. The score is the resulting belief.

* **Method 1, incoming messages:** the log-odds that neighbour `j`'s message adds to account `v`'s belief. Exact for the final iteration.
* **Method 2, leave-one-edge-out:** delete the edge `(v, j)`, run belief propagation again from scratch, report the change in `v`'s score.

**Correctness:** among the edges at `v`, do the high-importance ones lie inside its ring? (AUC over the incident edges, trolls only.)

This model is known to track node degree (section 3), so an honest explanation of it may mostly say "many neighbours". Each node needs about 10 belief-propagation re-runs, so this is the slowest section after the GNNs.

**Do not over-read the agreement row.** The belief is close to a monotone function of the summed messages, so removing an edge changes the score in about the same order as the edge's message. Both methods are two views of one calculation, and agreement near 1 is expected, not evidence of a trustworthy explanation. The informative columns are correctness and faithfulness.

In [ ]:
BPM = MODELS["belief_propagation"]
KS_BP = [1, 2, 3]
BP_SD = float(np.std(BP0[0][D.is_te]))
_BP_CACHE = {}


def bp_run(removed=()):
    keep = np.ones(E_UND, bool)
    keep[list(removed)] = False
    sc, log_m, a_idx, b_idx, _ = L.bp_scores(D, BPM, keep)
    return sc, log_m, b_idx, np.flatnonzero(keep)


def bp_node_score(v, removed=()):
    key = (v, tuple(sorted(int(e) for e in removed)))
    if key not in _BP_CACHE:
        _BP_CACHE[key] = float(bp_run(key[1])[0][v])
    return _BP_CACHE[key]


def bp_explain(v, removed=()):
    """(incident edge ids, message contributions, leave-one-edge-out changes) for node v in the graph minus `removed`."""
    sc, log_m, b_idx, kept = bp_run(removed)
    pos = np.flatnonzero(b_idx == v)
    ids = kept[pos % len(kept)]
    msg = log_m[pos, 1] - log_m[pos, 0]
    loo = np.array([sc[v] - bp_node_score(v, tuple(removed) + (int(e),)) for e in ids])
    return ids, msg, loo


def eval_belief_propagation():
    kinds = KIND_S[P_POS]
    ids, A = {}, {"Incoming messages": {}, "Leave-one-edge-out": {}}
    t0 = time.time()
    for v in P_NODES:
        ids[v], A["Incoming messages"][v], A["Leave-one-edge-out"][v] = bp_explain(v)
    secs = (time.time() - t0) / len(P_NODES)
    ok = [i for i, v in enumerate(P_NODES) if len(ids[v]) >= 2]          # need at least two edges to compare top vs random
    predict = lambda rows: np.array([bp_node_score(P_NODES[i], rem) for i, rem in rows])  # noqa: E731
    remove = lambda x, idx: (x[0], tuple(int(e) for e in ids[P_NODES[x[0]]][idx]))        # noqa: E731
    for m, d in A.items():
        auc = []
        for i, v in enumerate(P_NODES):
            if kinds[i] == "troll":
                truth = IS_RING[ids[v]]
                auc.append(roc_auc_score(truth, np.abs(d[v])) if 0 < truth.sum() < len(truth) else np.nan)
        SC_STRUCT.add("belief_propagation", m, "correctness_ring_edge_auc", auc, "chance = 0.5; trolls only; AUC over the edges at the account")
        vecs = [d[P_NODES[i]] for i in ok]
        dele = xm.deletion_gain(predict, [(i, ()) for i in ok], pad(vecs), remove, KS_BP, BP_SD, n_random=N_RANDOM_BP, seed=SEED,
                                n_items=np.array([len(ids[P_NODES[i]]) for i in ok]))
        faith = xm.mean_gain_per_instance(dele)
        dele.assign(node_index=dele["i"].map(dict(enumerate(ok))), model="belief_propagation", method=m).to_csv(
            xio.out_dir(D, FAMILY) / f"deletion_belief_propagation_{m.replace(' ', '_')}.csv", index=False)
        SC_STRUCT.add("belief_propagation", m, "faithfulness", faith)
        SC_STRUCT.add("belief_propagation", m, "seconds_per_explanation", secs)
    a, b = A["Incoming messages"], A["Leave-one-edge-out"]
    SC_STRUCT.add("belief_propagation", "messages vs leave-one-out", "agreement_spearman", [xm.rank_corr(np.abs(a[v]), np.abs(b[v])) for v in P_NODES if len(ids[v]) >= 2])
    SC_STRUCT.add("belief_propagation", "messages vs leave-one-out", "agreement_top3", [xm.topk_overlap(a[v], b[v], min(3, len(a[v]))) for v in P_NODES if len(ids[v]) >= 2])
    # ---- stability: drop one random incident edge outside the top 2, re-run, compare on the edges that remain
    r = np.random.default_rng(SEED + 5)
    st = {m: [] for m in A}
    for i in np.flatnonzero(np.isin(P_POS, STAB_POS_P)):
        v = P_NODES[i]
        if len(ids[v]) < 4:
            continue
        top2 = set(np.argsort(-np.abs(A["Incoming messages"][v]))[:2])
        j = int(r.choice([q for q in range(len(ids[v])) if q not in top2]))
        ids2, msg2, loo2 = bp_explain(v, removed=(int(ids[v][j]),))
        keep = [q for q in range(len(ids[v])) if q != j]
        st["Incoming messages"].append(xm.rank_corr(np.abs(A["Incoming messages"][v][keep]), np.abs(msg2)))
        st["Leave-one-edge-out"].append(xm.rank_corr(np.abs(A["Leave-one-edge-out"][v][keep]), np.abs(loo2)))
    for m, v in st.items():
        SC_STRUCT.add("belief_propagation", m, "stability_drop_item", v, "drop one incident edge outside the top 2; both methods are deterministic")
    STORE["belief_propagation"] = dict(attrs=A, ids=ids)
    display(SC_STRUCT.df[SC_STRUCT.df.model == "belief_propagation"].pivot_table(index="method", columns="criterion", values="value").round(3))

In [ ]:
print('=== belief_propagation ==='); eval_belief_propagation()

## 11. Scorecards

Two tables, one per sub-group. Higher is better for every column except `seconds_per_explanation`. Models are not ranked on a single number. Cells that are blank were not defined for that model (for example, correctness for the embedding models).

In [ ]:
order = ["correctness_ring_edge_auc", "correctness_own_feature_rank", "correctness_same_ring_top3", "faithfulness",
         "agreement_spearman", "agreement_top3", "agreement_top5", "agreement_own_features",
         "stability_noise", "stability_drop_item", "repeatability_seed", "seconds_per_explanation"]
for title, sc in [("Graph neural networks", SC_GNN), ("Structure-only models", SC_STRUCT)]:
    t = sc.table()
    print(title)
    display(t[[c for c in order if c in t.columns]].round(3))
print("Graph-reliance probe:")
display(PROBE.round(3))
print("What the embedding dimensions encode:")
display(EMB_PROBE.round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (crit, sc) in zip(axes, [("correctness_ring_edge_auc", SC_GNN), ("faithfulness", SC_GNN), ("faithfulness", SC_STRUCT)]):
    d = sc.df[sc.df.criterion == crit]
    d = d[~d.method.str.contains(" vs ")].reset_index(drop=True)
    y = np.arange(len(d))
    ax.barh(y, d["value"], 0.6, color="#4C78A8" if sc is SC_GNN else "#59A14F")
    ax.errorbar(d["value"], y, xerr=[(d["value"] - d["ci_low"]).clip(lower=0), (d["ci_high"] - d["value"]).clip(lower=0)], fmt="none", ecolor="black", lw=0.8, capsize=2)
    ax.set_yticks(y)
    ax.set_yticklabels(d["model"] + " / " + d["method"], fontsize=7)
    ax.invert_yaxis()
    ax.axvline(0.5 if "auc" in crit else 0, color="gray", lw=0.8)
    ax.set_title(crit.replace("_", " ") + (" (GNNs)" if sc is SC_GNN else " (structure-only)"))
plt.tight_layout()
plt.show()

In [ ]:
SC_ALL = xm.Scorecard("graph")
SC_ALL.rows = SC_GNN.rows + SC_STRUCT.rows
xio.save_scorecard(D, FAMILY, SC_ALL, reload_checks=CHECKS, meta=dict(
    seed=SEED, quick=QUICK, n_per_kind=N_PER_KIND, n_per_kind_prop=N_PER_KIND_PROP, gnnexp_epochs=GNNEXP_EPOCHS, ig_steps=IG_STEPS,
    ks_edges=KS_EDGES, ks_seeds=KS_SEEDS, ks_dims=KS_DIMS, pool=POOL, noise_frac=NOISE_FRAC,
    libraries={"shap": shap.__version__}))
xio.save_table(D, FAMILY, "reliance_probe", PROBE.reset_index())
xio.save_table(D, FAMILY, "embedding_probe", EMB_PROBE.reset_index())
pd.DataFrame(INST_ROWS).to_csv(xio.out_dir(D, FAMILY) / "per_instance_scores_gnn.csv", index=False)

## 12. What to take from this notebook

1. **GNNs.** Do GNNExplainer and Integrated Gradients find the ring edges of trolls (AUC above 0.5)? Does the answer depend on how much the model uses the graph (section 3)? Do the two methods agree on which edges matter?
2. **Structure-only.** For label spreading and belief propagation, the explanation is exact (the model's own arithmetic), so low faithfulness would be surprising; the interesting numbers are correctness (does the evidence come from the right ring?) and, for belief propagation, whether the "evidence" is just degree.
3. **Embeddings.** What do the probe results say about whether node2vec / DeepWalk dimensions can be explained to a person at all?
4. Where the same shortcut appears (degree), does any explanation reveal it?